---

<h3><center>E7 -  Introduction to Programming for Scientists and Engineers</center></h3>

<h2><center> Lecture w15-B<br></center></h2>
<h2><center>Monte Carlo simulation of a Lorenz attractor</h2>

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.stats import norm
import matplotlib.animation as animation
from IPython.display import HTML

\begin{align*}
x' &= \sigma(y-x) \\ 
y' &= x(\rho-z)-y \\
z' &= xy - \beta z
\end{align*}

## Global constants

In [ ]:
# physical parameters
sigma=10
rho=28
beta=8/3

# simulation parameters
T = 30       # [s] total simulation time
dt = 0.01    # [s] time time step

# initial condition
x0 = -10
y0 = -10
z0 = 30

## State equaton $\mathbf{y}' = f(t,\mathbf{y})$

In [ ]:
def f(t, Y):
    x, y, z = Y
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    return dxdt, dydt, dzdt

## Run SciPy's [`solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html)

In [ ]:
sol = solve_ivp(f, t_span=(0,T), 
                   y0=(x0,y0,z0), 
                   t_eval=np.arange(0,T,dt))

# unpack the solution
t = sol.t
x = sol.y[0,:]
y = sol.y[1,:]
z = sol.y[2,:]

In [ ]:
fig, ax = plt.subplots(figsize=(10,5),nrows=3,sharex=False)
ax[0].plot(t,x, color='b', linewidth=1)
ax[0].spines[['right','top']].set_visible(False)
ax[0].spines[['left','bottom']].set_position('zero')
ax[0].tick_params(axis='both',labelsize=12)

ax[1].plot(t,y, color='b', linewidth=1)
ax[1].spines[['right','top']].set_visible(False)
ax[1].spines[['left','bottom']].set_position('zero')
ax[1].tick_params(axis='both',labelsize=12)

ax[2].plot(t,z, color='b', linewidth=1)
ax[2].spines[['right','top']].set_visible(False)
ax[2].spines[['left','bottom']].set_position('zero')
ax[2].tick_params(axis='both',labelsize=12)

#######################

fig = plt.figure(figsize=(4,4))
ax = fig.add_subplot(111, projection='3d')
ax.plot(x, y, z, color='b', linewidth=0.5, alpha=0.7)
ax.set_xlim(-20,20)
ax.set_ylim(-20,20)

# Monte Carlo simulation

### Define distributions for the initial condition

In [ ]:

X0 = norm(loc=x0,scale=0.1)
Y0 = norm(loc=y0,scale=0.1)
Z0 = norm(loc=z0,scale=0.1)
n = 50

### Ensemble simulation

In [ ]:

# increase dt to save memory
dt = 0.15
t_eval = np.arange(0,T,dt)

# allocate
Xens = np.empty((len(t_eval),n))
Yens = np.empty((len(t_eval),n))
Zens = np.empty((len(t_eval),n))

# iterate over the ensemble
for i in range(n):

    # IVP solver
    sol = solve_ivp(f, t_span=(0,T), 
                    y0=(X0.rvs(),Y0.rvs(),Z0.rvs()), 
                    t_eval=t_eval)

    # unpack and store the solution
    Xens[:,i] = sol.y[0,:]
    Yens[:,i] = sol.y[1,:]
    Zens[:,i] = sol.y[2,:]

## Plots

### Plot initial condition

In [ ]:
fig = plt.figure(figsize=(4,4))
ax = fig.add_subplot(111, projection='3d')
ax.plot(x,y,z, color='b', linewidth=0.5, alpha=0.7)
ax.plot(x[0],y[0],z[0], marker='o', color='magenta')
ax.plot(Xens[0,:], Yens[0,:], Zens[0,:], linewidth=0,marker='.', color='r', markersize=3)
text = ax.text(0,-10,0, f"time = {t_eval[0]:.2f}",fontsize=12)
ax.set_xlim(-20,20)
ax.set_ylim(-20,20)
ax.set_zlim(0,40)
eps=10
ax.set_xlim(x0-eps,x0+eps)
ax.set_ylim(y0-eps,y0+eps)
ax.set_zlim(z0-eps,z0+eps)

### Plot time series

In [ ]:
fig, ax = plt.subplots(figsize=(10,5),nrows=3,sharex=False)
ax[0].plot(t_eval,Xens, linewidth=0.5)
ax[0].spines[['right','top']].set_visible(False)
ax[0].spines[['left','bottom']].set_position('zero')
ax[0].tick_params(axis='both',labelsize=12)

ax[1].plot(t_eval,Yens, linewidth=0.5)
ax[1].spines[['right','top']].set_visible(False)
ax[1].spines[['left','bottom']].set_position('zero')
ax[1].tick_params(axis='both',labelsize=12)

ax[2].plot(t_eval,Zens, linewidth=0.5)
ax[2].spines[['right','top']].set_visible(False)
ax[2].spines[['left','bottom']].set_position('zero')
ax[2].tick_params(axis='both',labelsize=12)

### Animate state space view

In [ ]:
fig = plt.figure(figsize=(4,4))
ax = fig.add_subplot(111, projection='3d')
ax.plot(x,y,z, color='b', linewidth=0.5, alpha=0.7)
point, = ax.plot(Xens[0,:], Yens[0,:], Zens[0,:], linewidth=0,marker='.', color='r', markersize=3)
text = ax.text(0,-10,0, f"time = {t_eval[0]:.2f}",fontsize=12)
ax.set_xlim(-20,20)
ax.set_ylim(-20,20)
ax.set_zlim(0,40)
plt.close()

def update(frame):
    point.set_data(Xens[frame,:], Yens[frame,:])
    point.set_3d_properties(Zens[frame,:])
    text.set_text(f"time = {t_eval[frame]:.2f}")
    
ani = animation.FuncAnimation(fig=fig, 
                                func=update, 
                                frames=len(t_eval), 
                                interval=100)

ani.save(filename='lorenz.gif', writer="pillow", fps=15)
HTML(ani.to_jshtml())